# 02 - Transformação

## Pipeline de Transformação e Engenharia de Atributos (02_transform)

### Visão Geral e Arquitetura do Pipeline

Este notebook implementa a camada de transformação (Silver) e a construção das Tabelas Analíticas de Modelagem (ABT - Gold). O objetivo é processar os dados brutos de crédito, consolidar a variável resposta e estruturar os dados para os experimentos de modelagem no notebook 03_modeling.

O pipeline opera inteiramente em memória para transformações intermediárias, persistindo em disco exclusivamente as bases consolidadas para modelagem.

---

### Fases do Pipeline

1. **Higienização e Tipagem Estrutural (Camada Silver):**
   - cadastral_tratada: relação 1:1 por cliente com consolidação por recência e fallback.
   - emprestimos_tratada: relação 1:N por contrato, com conservação volumétrica integral.
   - parcelas_tratada: relação 1:M por prestação, com remoção estrita de duplicatas exatas.

2. **Rollup de Parcelas e Construção do Target:**
   - Agrupamento gerando o DataFrame parcelas_por_contrato.
   - Consolidação de pagamentos fracionados com tolerância financeira.
   - Cálculo de dias de atraso (DPD) no horizonte de observação MOB 12.
   - Definição da target OVER60MOB12.

3. **ABT de Baseline (Benchmark sem Engenharia):**
   - Merge estrutural direto das bases tratadas com a target.
   - Sem criação de novas features derivadas.
   - Sem encoding, escalonamento ou imputações manuais.
   - Persistência em disco: **dados/abt_baseline.parquet**.

4. **Engenharia de Atributos (Feature Engineering):**
   - Features comportamentais históricas calculadas nas bases individuais (rollups por cliente).
   - Merge analítico completo.
   - Features de interação cruzadas pós-merge (idade na decisão, comprometimento de renda, razões financeiras).

5. **ABT Final para Modelagem:**
   - Validações de integridade, ausência de leakage e tipagem final.
   - Persistência em disco: dados/abt_modelagem.parquet.

6. **Preparação da Base de Inferência (Submissão):**
   - Aplicação estrita dos mesmos tratamentos cadastrais da base de treino.
   - Consolidação das features comportamentais históricas para a data de solicitação.
   - Persistência da base de escoragem para predição em produção.

---

### Diretrizes de Engenharia e Boas Práticas

- **Prevenção de Vazamento Temporal (Data Leakage):** Apenas informações disponíveis no momento da tomada de decisão (data_decisao no histórico e data_solicitacao na submissão) podem ser utilizadas como preditores. Variáveis de desfecho do contrato não entram como features.
- **Validação por Invariantes:** Asserções de pipeline (assert) devem testar consistência lógica e conservação volumétrica dinâmica entre entradas e saídas, evitando valores fixos (magic numbers).
- **Critério de Maturação da Safra (Target):**
  - **Evento Mau (1):** Contratos com atraso superior a 60 dias até a 12ª parcela.
  - **Evento Bom (0):** Contratos maduros (pelo menos 12 parcelas observadas ou com mais de 365 dias de histórico) sem atraso superior a 60 dias.
  - **Indeterminado (NA):** Contratos imaturos sem atraso crítico (excluídos do treinamento supervisionado).

In [ ]:
import sys
sys.path.append("..")

import src.functions as fn #Funcoes criadas no projeto
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Carrega cópias de trabalho mantendo os arquivos originais intactos.
cadastral = pd.read_parquet("../dados/base_cadastral.parquet").copy()
submissao = pd.read_parquet("../dados/base_submissao.parquet").copy()
emprestimos = pd.read_parquet("../dados/historico_emprestimos.parquet").copy()
parcelas = pd.read_parquet("../dados/historico_parcelas.parquet").copy()

print(
    f"cadastral: {cadastral.shape} | submissao: {submissao.shape} | "
    f"emprestimos: {emprestimos.shape} | parcelas: {parcelas.shape}"
)

## Tratamento de Dados para as tabelas-base de treino

In [ ]:
# --- TRATAMENTO E CONSOLIDAÇÃO: BASE CADASTRAL ---

# 1. Guarda referências dinâmicas para asserções
n_clientes_esperados = cadastral["id_cliente"].nunique()

# 2. Identificação de datas de nascimento sentinela e conversão de tipos
datas_convertidas = pd.to_datetime(cadastral["data_nascimento"], errors="coerce")
cadastral["data_nascimento_invalida"] = (
    cadastral["data_nascimento"].notna() & (datas_convertidas.dt.year <= 1920)
)
cadastral.loc[cadastral["data_nascimento_invalida"], "data_nascimento"] = None

# Aplica conversão padronizada de tipos estruturais
cadastral, rel_tipos = fn.converter_tipos(
    cadastral,
    datas={"data_nascimento": ["%Y-%m-%d", "%d/%m/%Y"]},
    numericas_texto={"renda_anual": {"decimal": ",", "milhar": "."}} if cadastral["renda_anual"].dtype == "object" else None,
    inteiros=["qtd_membros_familia"],
)

# 3. Padronização de variáveis categóricas e conversão de placeholders em NaN
cols_categoricas = [
    c for c in cadastral.select_dtypes(include=["object", "string", "category"]).columns 
    if c not in ["data_nascimento"]
]
cadastral, rel_cat = fn.padronizar_categoricas(cadastral, colunas=cols_categoricas)

# 4. Consolidação de duplicatas: recência com preenchimento de nulos
mascara_dup = cadastral.duplicated(subset=["id_cliente"], keep=False)
cad_unicos = cadastral.loc[~mascara_dup].set_index("id_cliente")

cad_1a = cadastral.loc[cadastral.duplicated(subset=["id_cliente"], keep="last")].set_index("id_cliente")
cad_2a = cadastral.loc[cadastral.duplicated(subset=["id_cliente"], keep="first")].set_index("id_cliente")

# A 2ª ocorrência prevalece; onde houver NaN, herda o valor da 1ª ocorrência
cad_dups_consolidadas = cad_2a.combine_first(cad_1a)

cadastral_tratada = (
    pd.concat([cad_unicos, cad_dups_consolidadas])
    .sort_index()
    .reset_index()
)

# 5. Otimização de tipos finais para controle de memória
cadastral_tratada["id_cliente"] = cadastral_tratada["id_cliente"].astype("int64")
cadastral_tratada["qtd_filhos"] = cadastral_tratada["qtd_filhos"].astype("Int64")
for col in cols_categoricas:
    cadastral_tratada[col] = cadastral_tratada[col].astype("category")

# 6. Remoção de colunas sensíveis/não utilizadas no modelo
cols_para_remover = ["sexo"]
n_cols_antes_drop = cadastral_tratada.shape[1]
cadastral_tratada = cadastral_tratada.drop(columns=cols_para_remover)

# 7. Asserções para validação do pipeline
assert cadastral_tratada.shape[0] == n_clientes_esperados, (
    f"Esperado {n_clientes_esperados} linhas, obtido {cadastral_tratada.shape[0]}"
)
assert cadastral_tratada["id_cliente"].nunique() == n_clientes_esperados, (
    "Falha: id_cliente não é único na base consolidada"
)
assert cadastral_tratada.shape[1] == (n_cols_antes_drop - len(cols_para_remover)), (
    "Inconsistência no número de colunas após exclusão de campos indesejados"
)

print("--- Base Cadastral Tratada com Sucesso ---")
print(f"Shape final: {cadastral_tratada.shape}")
print(f"Clientes únicos garantidos: {cadastral_tratada['id_cliente'].nunique()}")
print(f"Total com data_nascimento_invalida: {cadastral_tratada['data_nascimento_invalida'].sum()}")

In [ ]:
cadastral_tratada.head()

In [ ]:
# --- TRATAMENTO E TIPAGEM: BASE EMPRESTIMOS ---

# 1. Guarda referências dinâmicas da base de entrada
n_linhas_esperadas_emp = len(emprestimos)
n_contratos_esperados_emp = emprestimos["id_contrato"].nunique()

emprestimos_proc = emprestimos.copy()

# 2. Conversão de colunas temporais
datas_emprestimos = {
    "data_decisao": ["%Y-%m-%d", "%d/%m/%Y"],
    "data_primeiro_vencimento": ["%Y-%m-%d", "%d/%m/%Y"],
}

emprestimos_proc, rel_datas = fn.converter_tipos(
    emprestimos_proc,
    datas={c: datas_emprestimos[c] for c in datas_emprestimos if c in emprestimos_proc.columns},
)

# 3. Padronização de colunas categóricas e conversão de placeholders em nulo
cols_cat_emp = [
    c for c in emprestimos_proc.select_dtypes(include=["object", "string", "category"]).columns
    if c not in datas_emprestimos
]

emprestimos_proc, rel_cat_emp = fn.padronizar_categoricas(
    emprestimos_proc, colunas=cols_cat_emp
)

# 4. Otimização de tipos de dados
emprestimos_proc["id_contrato"] = emprestimos_proc["id_contrato"].astype("int64")
emprestimos_proc["id_cliente"] = emprestimos_proc["id_cliente"].astype("int64")

for col in cols_cat_emp:
    emprestimos_proc[col] = emprestimos_proc[col].astype("category")

emprestimos_tratada = emprestimos_proc.copy()

# 7. Asserções para validação do pipeline
assert len(emprestimos_tratada) == n_linhas_esperadas_emp, (
    f"Perda de registros: esperado {n_linhas_esperadas_emp}, obtido {len(emprestimos_tratada)}"
)
assert emprestimos_tratada["id_contrato"].nunique() == n_contratos_esperados_emp, (
    f"Inconsistência de chave: esperado {n_contratos_esperados_emp} contratos únicos, "
    f"obtido {emprestimos_tratada['id_contrato'].nunique()}"
)

print("--- Base Empréstimos Tratada com Sucesso ---")
print(f"Shape final: {emprestimos_tratada.shape}")
print(f"Contratos únicos: {emprestimos_tratada['id_contrato'].nunique()}")
print(f"Clientes distintos na base: {emprestimos_tratada['id_cliente'].nunique()}")

In [ ]:
emprestimos_tratada.head()

In [ ]:
# --- TRATAMENTO E HIGIENIZAÇÃO: BASE PARCELAS ---

# 1. Guarda referências dinâmicas da base de entrada
n_linhas_originais_parc = len(parcelas)
n_duplicadas_integrais = int(parcelas.duplicated().sum())
n_linhas_esperadas_parc = n_linhas_originais_parc - n_duplicadas_integrais

parcelas_proc = parcelas.copy()

# 2. Remoção exclusiva de duplicatas integrais exatas
parcelas_proc = parcelas_proc.drop_duplicates()

# 3. Conversão de colunas temporais
datas_parcelas = {
    "data_prevista_pagamento": ["%Y-%m-%d", "%d/%m/%Y"],
    "data_pagamento": ["%Y-%m-%d", "%d/%m/%Y"],
}

parcelas_proc, rel_datas_parc = fn.converter_tipos(
    parcelas_proc,
    datas={c: datas_parcelas[c] for c in datas_parcelas if c in parcelas_proc.columns},
)

# 4. Otimização de tipos estruturais
parcelas_proc["id_contrato"] = parcelas_proc["id_contrato"].astype("int64")

if "versao_parcela" in parcelas_proc.columns:
    parcelas_proc["versao_parcela"] = parcelas_proc["versao_parcela"].astype("Int64")

if "numero_parcela" in parcelas_proc.columns:
    parcelas_proc["numero_parcela"] = parcelas_proc["numero_parcela"].astype("Int64")

parcelas_tratada = parcelas_proc.copy()

# 5. Asserções para validação do pipeline
assert len(parcelas_tratada) == n_linhas_esperadas_parc, (
    f"Inconsistência volumétrica: esperado {n_linhas_esperadas_parc} linhas, "
    f"obtido {len(parcelas_tratada)}"
)
assert parcelas_tratada.duplicated().sum() == 0, (
    "Falha: ainda existem duplicatas integrais na base de parcelas"
)

print("--- Base Parcelas Tratada com Sucesso ---")
print(f"Linhas originais: {n_linhas_originais_parc}")
print(f"Duplicatas integrais removidas: {n_duplicadas_integrais}")
print(f"Shape final: {parcelas_tratada.shape}")
print(f"Contratos distintos com histórico de parcelas: {parcelas_tratada['id_contrato'].nunique()}")

In [ ]:
parcelas_tratada.head()

## Documentação de Decisões: Higienização e Tipagem das Bases Históricas

Esta etapa estabelece a camada de dados limpos (*Silver*) para o desenvolvimento do modelo, padronizando formatos, resolvendo inconsistências estruturais e garantindo integridade relacional sem vazamento de dados. A base de inferência (`submissao`) é preservada para avaliação posterior.

---

### 1. Base `cadastral` (`cadastral_tratada`)

- **Datas Sentinela e Inconsistentes:**
  - Identificação de anos de nascimento irrealistas (ano <= 1920).
  - Criação da flag booleana `data_nascimento_invalida` para rastreabilidade de negócio.
  - Conversão dos valores anômalos em `data_nascimento` para nulo (`pd.NaT`).
- **Conversão de Tipos Estruturais:**
  - `data_nascimento`: convertida para `datetime64[ns]` via múltiplos formatos (`%Y-%m-%d`, `%d/%m/%Y`).
  - `renda_anual`: higienização textual de moedas e conversão para `float64`.
  - `qtd_membros_familia`: convertida para inteiro com suporte a nulos (`Int64`).
- **Padronização Categórica e Placeholders:**
  - Remoção de acentuação, padronização em minúsculas e eliminação de espaços duplicados em todas as variáveis textuais.
  - Conversão de placeholders de ausência disfarçada (ex.: `"não informado"`, `"unknown"`, `"-"`) em `NaN` via `fn.padronizar_categoricas`.
- **Deduplicação e Atualização Cadastral:**
  - Clientes repetidos consolidados via `.combine_first()`.
  - A 2ª ocorrência (registro mais recente) prevalece como fonte primária; valores ausentes herdam os dados preenchidos na 1ª ocorrência.
- **Otimização de Memória e Conformidade:**
  - Conversão de `id_cliente` para `int64`, `qtd_filhos` para `Int64` e colunas textuais para `category`.
  - Remoção da coluna sensível `sexo` da base de modelagem para assegurar equidade algorítmica (*fairness*).
- **Validação de Integridade (Asserções Dinâmicas):**
  - Relação 1:1 estrita: total de linhas final igual ao número de `id_cliente` únicos originais (50.000).

---

### 2. Base `emprestimos` (`emprestimos_tratada`)

- **Preservação Volumétrica e Integridade Histórica:**
  - Manutenção de todos os contratos e histórico original (211.944 linhas), sem exclusão de linhas nem engenharia antecipada.
- **Conversão Temporal:**
  - `data_decisao` e `data_primeiro_vencimento`: convertidas para `datetime64[ns]`.
- **Padronização Categórica:**
  - Higienização textual e conversão de placeholders em `NaN` em atributos como `tipo_contrato`, `status_contrato`, etc.
- **Otimização de Memória e Tipagem:**
  - `id_contrato` e `id_cliente`: tipificados como `int64`.
  - Colunas categóricas textuais convertidas para `category`.
- **Validação de Integridade (Asserções Dinâmicas):**
  - Conservação exata da volumetria de entrada (`len(emprestimos_tratada) == len(emprestimos)`).
  - Unicidade estrita da chave primária `id_contrato` comprovada via assert.

---

### 3. Base `parcelas` (`parcelas_tratada`)

- **Eliminação de Duplicatas Técnicas:**
  - Remoção estrita de duplicatas integrais exatas (`drop_duplicates()`), preservando pagamentos parciais e amortizações legítimas.
- **Conversão Temporal:**
  - `data_prevista_pagamento` e `data_pagamento`: padronizadas para `datetime64[ns]`.
- **Otimização Estrutural e Tipagem:**
  - `id_contrato`: tipificado como `int64`.
  - `versao_parcela` e `numero_parcela`: convertidos para inteiros anuláveis (`Int64`).
- **Validação de Integridade (Asserções Dinâmicas):**
  - Conservação volumétrica exata calculada dinamicamente: `n_linhas_originais - n_duplicadas_integrais`.
  - Assert garantindo zero duplicatas integrais remanescentes na saída.

---

### Resumo Comparativo de Contratos de Saída

| Base | Chave Primária | Relação | Linhas Originais | Linhas Finais | Ação Principal de Limpeza |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **`cadastral_tratada`** | `id_cliente` | 1 : 1 | 50.600 | 50.000 | Consolidação por recência + fallback; exclusão de `sexo`; flag de nascimento. |
| **`emprestimos_tratada`** | `id_contrato` | 1 : N (cliente) | 211.944 | 211.944 | Conversão de datas e categorias; conservação estrita de contratos. |
| **`parcelas_tratada`** | Composta (`id_contrato`, `versao`, `num`) | 1 : M (contrato) | 1.664.384 | 1.659.408 | Exclusão de 4.976 duplicatas integrais; tipagem temporal e de contadores. |

## Criação da target 

### OVER60MOB12